Final project

An automated data pipeline for pollen, air pollution and weather in Swedish cities, with a dashboard that shows when and where pollen levels are highest, and why.

## Why this analysis? Pollen allergy in Sweden in numbers

| Fact | Number | Official source |
|---|---|---|
| **How common is it?** | About **1 in 4** people in Sweden get pollen-related symptoms in spring and summer; up to **30%** of the population is estimated to be allergic to pollen | [1177](https://www.1177.se/sjukdomar--besvar/allergier-och-overkanslighet/pollenallergi/) (the regions' official healthcare guide); [CAMM 2026](#sources), p. 24 |
| **Stockholm County** | **43%** of adults report pollen allergy (2023), up from **30%** in 2007 (+13 percentage points); women 45%, men 41% | Miljöhälsoenkäten 2023 (Folkhälsomyndigheten), reported in [CAMM Miljöhälsorapport 2025](#sources), ch. 12 |
| **Most common allergy** | "Pollen allergy is the most common allergy and the one that has increased most in recent decades" | [CAMM Miljöhälsorapport 2025](#sources), ch. 12 |
| **Main allergens** | Birch and grass cause the most reactions; also mugwort, alder and hazel | [1177](https://www.1177.se/sjukdomar--besvar/allergier-och-overkanslighet/pollenallergi/) |
| **Seasons start earlier and last longer** | Compared with the early 1970s in Stockholm, birch starts **~16 days earlier**; the grass season is **~19 days longer** and mugwort **~40 days longer** | [CAMM 2026](#sources), p. 17 and Table B1 |
| **More high-pollen days** | ~**10 more days per season** with high birch pollen in 2019–2024 than in 1973–1977 | [CAMM 2026](#sources), p. 18 |
| **Load on health care** | Outpatient visits for respiratory symptoms in the Stockholm area rise from ~**267/day** (no birch pollen) to ~**390/day** (high birch pollen); the extra high-pollen days ≈ **530–540 extra visits per season** (illustrative estimate) | [CAMM 2026](#sources), p. 18 |
| **Cost to society** | ~**13 billion SEK per year** (2016 estimate) for pollen-related health effects in Sweden, mostly **indirect** (lost productivity, sick leave) | [CAMM 2026](#sources), p. 13 (citing Cardell et al. 2016) |
| **Long-lasting** | ~**75%** of children with pollen-induced hay fever still have symptoms as young adults, and nearly **1 in 3** develop asthma | [CAMM 2026](#sources), p. 26 (BAMSE cohort) |
| **Timing matters for treatment** | Over-the-counter medicine works best if started **1–2 weeks before the season** begins | [1177](https://www.1177.se/sjukdomar--besvar/allergier-och-overkanslighet/pollenallergi/) |

**Why this project:** millions of people in Sweden are affected, seasons are getting earlier and longer, and treatment works best when started **before** the season. But official pollen measurements come from a few stations and are published as forecasts or reports. This project builds an **automatic pipeline** that answers **when and where** pollen is high – for **6 cities from south to north**, **every hour**, for **all four main pollen types**, together with **weather and air quality** – so that people with allergies can plan, and businesses (pharmacies, retail, tourism, events) can act at the right time and place.

<a id="sources"></a>**Sources**
- **CAMM 2026:** Lõhmus Sundström M, Stucki L, Andersson N, Ekebom A, Gedda B, Jonsson M et al. *Förändringar i pollensäsonger, klimat och folkhälsa.* Centrum för arbets- och miljömedicin (CAMM), Region Stockholm, 2026. Measurements from the Palynological Laboratory, Swedish Museum of Natural History, 1973–2024. [PDF](https://www.folkhalsoguiden.se/49c72d/globalassets/verksamheter/forskning-och-utveckling/centrum-for-arbets--och-miljomedicin1/rapporter/pollenrapport_klar.pdf)
- **CAMM Miljöhälsorapport 2025**, chapter 12 *Allergi och sjukdomar i luftvägarna*, Region Stockholm – based on Miljöhälsoenkäten (Folkhälsomyndigheten) 2007, 2015 and 2023. [Link](https://www.camm.regionstockholm.se/rapporter-och-faktablad/rapporter/miljohalsorapporter/miljohalsorapport-2025/12-allergi-och-sjukdomar-i-luftvagarna/)
- **1177**, *Pollenallergi* – the Swedish regions' official healthcare guide. [Link](https://www.1177.se/sjukdomar--besvar/allergier-och-overkanslighet/pollenallergi/)

Creating a workspace: 

Do a folder in the github folder - best practice for the project works well with all the uv add

Make sure to create a venv and then in terminal add the uv init and uv add

to create i start: uv init, uv add y uv sync (this last one dpeends when might not matck uv lock)

In [ ]:
# Setup: run this first after every kernel restart (imports only, no database connection)
from pathlib import Path

import altair as alt
import duckdb
import pandas as pd

# Project root = first folder upwards that contains pyproject.toml (works from notebooks/ or the root)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
DB_PATH = str(ROOT / "data" / "polen.duckdb")

Step 1

Testing if we can retrieve data:

Source: [Open-Meteo Air Quality API](https://open-meteo.com/en/docs/air-quality-api) (CAMS European model, ~11 km resolution). No API key required; free for non-commercial use.

Phase 0 – validate the data before building the pipeline:
1. Test call for one city (Stockholm)
2. Check that all 6 cities return pollen and pollution
3. Find the first date with pollen data → defines the project period

In [ ]:
import truststore
truststore.inject_into_ssl()  # use Windows certificates (Avast HTTPS scanning breaks Python's default ones)

import requests
import pandas as pd

URL = "https://air-quality-api.open-meteo.com/v1/air-quality"

HOURLY_VARS = [
    "alder_pollen", "birch_pollen", "grass_pollen", "mugwort_pollen",
    "pm2_5", "pm10", "ozone", "nitrogen_dioxide",
]

# Cities from south to north (latitude, longitude)
CITIES = {
    "Malmö":     (55.605, 13.003),
    "Göteborg":  (57.709, 11.974),
    "Stockholm": (59.329, 18.069),
    "Uppsala":   (59.858, 17.639),
    "Umeå":      (63.826, 20.263),
    "Luleå":     (65.584, 22.155),
}

In [ ]:
def get_air_quality(lat, lon, start_date, end_date, variables=HOURLY_VARS):
    """Download hourly air quality data for one location and return a DataFrame."""
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": ",".join(variables),
        "start_date": start_date,
        "end_date": end_date,
        "timezone": "Europe/Stockholm",
    }
    response = requests.get(URL, params=params, timeout=60)
    response.raise_for_status()  # stop with a clear error if status is not 200

    df = pd.DataFrame(response.json()["hourly"])  # one row per hour
    df["time"] = pd.to_datetime(df["time"])
    return df


# 1. Test call: Stockholm, May 2024
stockholm = get_air_quality(*CITIES["Stockholm"], "2024-05-01", "2024-05-31")
print(stockholm.shape)
stockholm.head()

(744, 9)


,time,alder_pollen,birch_pollen,grass_pollen,mugwort_pollen,pm2_5,pm10,ozone,nitrogen_dioxide
0,2024-05-01 00:00:00,0.0,115.9,0.0,0.0,8.7,12.9,40.0,33.4
1,2024-05-01 01:00:00,0.0,100.8,0.0,0.0,8.5,11.7,34.0,30.2
2,2024-05-01 02:00:00,0.0,124.4,0.1,0.0,8.8,11.3,41.0,32.0
3,2024-05-01 03:00:00,0.0,107.4,0.1,0.0,8.6,10.5,44.0,32.2
4,2024-05-01 04:00:00,0.0,99.3,0.0,0.0,8.0,9.6,52.0,31.3


In [ ]:
# 2. All 6 cities: share of missing values per variable (0 = complete)
frames = []
for city, (lat, lon) in CITIES.items():
    city_df = get_air_quality(lat, lon, "2024-05-01", "2024-05-31")
    city_df["city"] = city
    frames.append(city_df)

all_cities = pd.concat(frames, ignore_index=True)

all_cities.drop(columns="time").groupby("city", sort=False).agg(lambda s: s.isna().mean()).round(2)

,alder_pollen,birch_pollen,grass_pollen,mugwort_pollen,pm2_5,pm10,ozone,nitrogen_dioxide
city,,,,,,,,
Malmö,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Göteborg,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Stockholm,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Uppsala,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Umeå,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
Luleå,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# 3. First date with pollen data (Stockholm, 2019–2022)
pollen_vars = ["alder_pollen", "birch_pollen", "grass_pollen", "mugwort_pollen"]
history = get_air_quality(*CITIES["Stockholm"], "2019-01-01", "2022-12-31", variables=pollen_vars)

history.set_index("time")[pollen_vars].apply(lambda s: s.first_valid_index())

alder_pollen     2020-11-25 02:00:00
birch_pollen     2020-10-07 02:00:00
grass_pollen     2020-10-07 02:00:00
mugwort_pollen   2021-06-09 02:00:00
dtype: datetime64[us]

Step 2 - Ingestion (`ingestion/ingest.py`)

Run: `uv run python ingestion/ingest.py`

Sources (no API key):
- Pollen + pollution: [Open-Meteo Air Quality API](https://air-quality-api.open-meteo.com/v1/air-quality) (CAMS)
- Weather: [Open-Meteo Historical Weather API](https://archive-api.open-meteo.com/v1/archive)

Steps:
1. Check last stored date per city → download only new days (from 2021-01-01 on first run)
2. Call the API (retries on errors)
3. Save to DuckDB `data/polen.duckdb` → tables `raw.air_quality`, `raw.weather` (UTC, no duplicates)
4. Log each run in `raw.ingestion_log`

Settings (cities, variables, dates): `ingestion/config.py`

Step 3 - Explore raw data (schema `raw` in `data/polen.duckdb`)

In [1]:
import duckdb
import pandas as pd



In [ ]:
# read_only=True: explore without locking the file for ingest.py
con = duckdb.connect(DB_PATH, read_only=True)

con.sql("SHOW ALL TABLES")

In [ ]:
# SQL result -> pandas DataFrame (times in UTC)
df = con.sql("""
    SELECT *
    FROM raw.air_quality
    WHERE city = 'Stockholm'
      AND time BETWEEN '2024-05-01' AND '2024-05-31'
    ORDER BY time
""").df()
df.head()

,city,time,alder_pollen,birch_pollen,grass_pollen,mugwort_pollen,pm2_5,pm10,ozone,nitrogen_dioxide,loaded_at
0,Stockholm,2024-05-01 00:00:00,0.0,124.4,0.1,0.0,8.8,11.3,41.0,32.0,2026-10-05 11:26:23.515184
1,Stockholm,2024-05-01 01:00:00,0.0,107.4,0.1,0.0,8.6,10.5,44.0,32.2,2026-10-05 11:26:23.515184
2,Stockholm,2024-05-01 02:00:00,0.0,99.3,0.0,0.0,8.0,9.6,52.0,31.3,2026-10-05 11:26:23.515184
3,Stockholm,2024-05-01 03:00:00,0.0,65.5,0.0,0.0,7.4,9.9,59.0,26.8,2026-10-05 11:26:23.515184
4,Stockholm,2024-05-01 04:00:00,0.0,56.3,0.0,0.0,6.9,10.4,64.0,22.5,2026-10-05 11:26:23.515184


In [ ]:
# Pipeline log: one row per city and source per run
con.sql("SELECT * FROM raw.ingestion_log ORDER BY run_at DESC")

┌────────────────────────────┬─────────────┬───────────┬────────────┬────────────┬─────────────┬─────────┬─────────┐
│           run_at           │   source    │   city    │ start_date │  end_date  │ rows_loaded │ status  │  error  │
│         timestamp          │   varchar   │  varchar  │    date    │    date    │    int32    │ varchar │ varchar │
├────────────────────────────┼─────────────┼───────────┼────────────┼────────────┼─────────────┼─────────┼─────────┤
│ 2026-10-05 11:28:21.717571 │ weather     │ Luleå     │ 2021-01-01 │ 2026-10-04 │       50472 │ ok      │ NULL    │
│ 2026-10-05 11:28:19.42227  │ air_quality │ Luleå     │ 2021-01-01 │ 2026-10-04 │       50472 │ ok      │ NULL    │
│ 2026-10-05 11:28:17.660905 │ weather     │ Umeå      │ 2021-01-01 │ 2026-10-04 │       50472 │ ok      │ NULL    │
│ 2026-10-05 11:28:16.026143 │ air_quality │ Umeå      │ 2021-01-01 │ 2026-10-04 │       50472 │ ok      │ NULL    │
│ 2026-10-05 11:28:14.590838 │ weather     │ Uppsala   │ 2021-01

Data overview

| | Value |
|---|---|
| **Cities** (south → north) | Malmö, Göteborg, Stockholm, Uppsala, Umeå, Luleå |
| **Granularity** | City level: one point (lat, lon) per city, ~11 km model grid cell. Not kommun level |
| **Frequency** | Hourly (1 row per city per hour) |
| **Period** | 2021-01-01 00:00 → yesterday (UTC), ~5.8 years |
| **Rows per city** | ~50.5K hours |
| **Total** | 6 cities × ~50.5K = ~302.8K rows per table (`air_quality`, `weather`) |
| **Pollen** | Complete within each season; off-season gaps until 2023 |

Variables explained

| Variable | What it is (simple) | Unit |
|---|---|---|
| `alder_pollen` | Pollen from alder trees (al), the first of the year | grains/m³ |
| `birch_pollen` | Pollen from birch trees (björk), the strongest allergy trigger in Sweden | grains/m³ |
| `grass_pollen` | Pollen from grasses (gräs) | grains/m³ |
| `mugwort_pollen` | Pollen from mugwort (gråbo), a weed, late summer | grains/m³ |
| `pm2_5` | Very small dust particles, go deep into the lungs | µg/m³ |
| `pm10` | Small dust particles (road dust, smoke) | µg/m³ |
| `ozone` | Gas formed by sunlight + pollution; irritates airways | µg/m³ |
| `nitrogen_dioxide` | Gas mainly from car traffic | µg/m³ |
| `temperature_2m` | Air temperature, 2 m above the ground | °C |
| `relative_humidity_2m` | How wet the air is (100% = saturated) | % |
| `precipitation` | Rain or snow in that hour | mm |
| `wind_speed_10m` | Wind speed, 10 m above the ground | km/h |
| `wind_direction_10m` | Where the wind comes from (0° = north, 90° = east) | ° |

- **grains/m³** = how many pollen grains are in one cubic metre of air
- **µg/m³** = micrograms (millionths of a gram) per cubic metre of air
- Pollen values are model estimates (CAMS), not measurements

Tables in `data/polen.duckdb` (schema `raw`)

| Table | Source | Content | Key (unique) | Columns |
|---|---|---|---|---|
| `raw.air_quality` | [Open-Meteo Air Quality API](https://open-meteo.com/en/docs/air-quality-api) (CAMS European model) | Pollen + pollution per city and hour | `city` + `time` | `alder_pollen`, `birch_pollen`, `grass_pollen`, `mugwort_pollen` (grains/m³); `pm2_5`, `pm10`, `ozone`, `nitrogen_dioxide` (µg/m³) |
| `raw.weather` | [Open-Meteo Historical Weather API](https://open-meteo.com/en/docs/historical-weather-api) | Weather per city and hour | `city` + `time` | `temperature_2m` (°C), `relative_humidity_2m` (%), `precipitation` (mm), `wind_speed_10m` (km/h), `wind_direction_10m` (°) |
| `raw.ingestion_log` | Generated by `ingestion/ingest.py` | One row per city and source per pipeline run | None (log) | `run_at`, `source`, `city`, `start_date`, `end_date`, `rows_loaded`, `status`, `error` |

- `time` is in UTC; `loaded_at` = when the row was saved
- Same key in `air_quality` and `weather` → they join on `city` + `time`

In [ ]:
# Period and length per city (n_hours = expected_hours → no missing hours)
con.sql("""
    SELECT city,
           min(time) AS first_hour,
           max(time) AS last_hour,
           count(*)  AS n_hours,
           date_diff('hour', min(time), max(time)) + 1 AS expected_hours
    FROM raw.air_quality
    GROUP BY city
    ORDER BY min(time), city
""")

┌───────────┬─────────────────────┬─────────────────────┬─────────┬────────────────┐
│   city    │     first_hour      │      last_hour      │ n_hours │ expected_hours │
│  varchar  │      timestamp      │      timestamp      │  int64  │     int64      │
├───────────┼─────────────────────┼─────────────────────┼─────────┼────────────────┤
│ Göteborg  │ 2021-01-01 00:00:00 │ 2026-10-04 23:00:00 │   50472 │          50472 │
│ Luleå     │ 2021-01-01 00:00:00 │ 2026-10-04 23:00:00 │   50472 │          50472 │
│ Malmö     │ 2021-01-01 00:00:00 │ 2026-10-04 23:00:00 │   50472 │          50472 │
│ Stockholm │ 2021-01-01 00:00:00 │ 2026-10-04 23:00:00 │   50472 │          50472 │
│ Umeå      │ 2021-01-01 00:00:00 │ 2026-10-04 23:00:00 │   50472 │          50472 │
│ Uppsala   │ 2021-01-01 00:00:00 │ 2026-10-04 23:00:00 │   50472 │          50472 │
└───────────┴─────────────────────┴─────────────────────┴─────────┴────────────────┘

Missing values (% of hours with NULL)

In [ ]:
# % missing per city and variable, for both tables
# count(column) counts only non-NULL values -> 1 - count/total = share missing
# COLUMNS(...) applies the same formula to every variable column
for table in ["air_quality", "weather"]:
    print(table)
    display(con.sql(f"""
        SELECT city,
               round(100 * (1 - count(COLUMNS(* EXCLUDE (city, time, loaded_at))) / count(*)), 1)
        FROM raw.{table}
        GROUP BY city
        ORDER BY city
    """).df())

air_quality


,city,alder_pollen,birch_pollen,grass_pollen,mugwort_pollen,pm2_5,pm10,ozone,nitrogen_dioxide
0,Göteborg,24.2,25.3,20.9,28.5,0.0,0.0,0.0,0.0
1,Luleå,24.2,25.3,20.9,28.5,0.0,0.0,0.0,0.0
2,Malmö,24.2,25.3,20.9,28.5,0.0,0.0,0.0,0.0
3,Stockholm,24.2,25.3,20.9,28.5,0.0,0.0,0.0,0.0
4,Umeå,24.2,25.3,20.9,28.5,0.0,0.0,0.0,0.0
5,Uppsala,24.2,25.3,20.9,28.5,0.0,0.0,0.0,0.0


weather


,city,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m
0,Göteborg,0.0,0.0,0.0,0.0,0.0
1,Luleå,0.0,0.0,0.0,0.0,0.0
2,Malmö,0.0,0.0,0.0,0.0,0.0
3,Stockholm,0.0,0.0,0.0,0.0,0.0
4,Umeå,0.0,0.0,0.0,0.0,0.0
5,Uppsala,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Pollen: % missing per year -> where are the gaps?
con.sql("""
    SELECT year(time) AS year,
           round(100 * (1 - count(COLUMNS('.*_pollen')) / count(*)), 1)
    FROM raw.air_quality
    GROUP BY year
    ORDER BY year
""")

┌───────┬──────────────┬──────────────┬──────────────┬────────────────┐
│ year  │ alder_pollen │ birch_pollen │ grass_pollen │ mugwort_pollen │
│ int64 │    double    │    double    │    double    │     double     │
├───────┼──────────────┼──────────────┼──────────────┼────────────────┤
│  2021 │         50.4 │         41.9 │         33.4 │           60.3 │
│  2022 │         50.4 │         58.1 │         49.6 │           58.1 │
│  2023 │         38.3 │         46.0 │         37.5 │           46.0 │
│  2024 │          0.0 │          0.0 │          0.0 │            0.0 │
│  2025 │          0.0 │          0.0 │          0.0 │            0.0 │
│  2026 │          0.0 │          0.0 │          0.0 │            0.0 │
└───────┴──────────────┴──────────────┴──────────────┴────────────────┘

Result:
- **No missing hours** (n_hours = expected_hours for every city)
- **Weather and pollution** (PM2.5, PM10, ozone, NO2): 0% missing
- **Pollen**: ~21–29% missing, only in 2021–2023 and only outside each pollen season (CAMS did not publish off-season values then). Since 2024: 0%
- Off-season NULL → filled with 0 in staging ✅ (see Decisions for Phase 2)

Data profiling (input for Phase 2)

Goal: check value ranges, when each pollen season happens, and typical daily levels → to define the "season start" threshold.

In [ ]:
# 1. Value ranges: any negatives or impossible values?
for table in ["air_quality", "weather"]:
    print(table)
    display(con.sql(f"SUMMARIZE raw.{table}").df()
            .query("column_name not in ['city', 'time', 'loaded_at']")
            [["column_name", "min", "max", "avg", "q50"]])

air_quality


,column_name,min,max,avg,q50
2,alder_pollen,0.0,1760.6,0.6110363713940341,0.0
3,birch_pollen,0.0,3060.2,20.85790274497375,0.0
4,grass_pollen,0.0,107.3,2.331838415219895,0.0
5,mugwort_pollen,0.0,48.0,0.5120990768718634,0.0
6,pm2_5,0.4,95.1,4.987334231521117,3.9808592950814203
7,pm10,0.5,113.6,7.8077789005125116,6.581413833327922
8,ozone,0.0,146.0,58.43681975484757,58.99879800362227
9,nitrogen_dioxide,0.1,72.2,6.1369224520526355,4.250960112279801


weather


,column_name,min,max,avg,q50
2,temperature_2m,-32.0,33.1,7.080140143710059,7.32916439467626
3,relative_humidity_2m,12.0,100.0,78.94631016537222,82.67211643915536
4,precipitation,0.0,22.9,0.08760533893380873,0.0
5,wind_speed_10m,0.0,61.0,13.689434736091071,12.572467227246156
6,wind_direction_10m,0.0,360.0,200.92463147883976,211.87757313557927


In [ ]:
# 2. Daily mean pollen per city, in Swedish local time (UTC -> Europe/Stockholm)
POLLEN = ["alder_pollen", "birch_pollen", "grass_pollen", "mugwort_pollen"]

daily = con.sql("""
    SELECT city,
           CAST(timezone('Europe/Stockholm', time AT TIME ZONE 'UTC') AS DATE) AS day,
           avg(COLUMNS('.*_pollen'))
    FROM raw.air_quality
    GROUP BY ALL
    ORDER BY city, day
""").df()
daily["day"] = pd.to_datetime(daily["day"])

# Season months: average per month, only complete years without gaps (2024–2025)
complete = daily[daily["day"].dt.year.isin([2024, 2025])]
complete.groupby(complete["day"].dt.month)[POLLEN].mean().round(1).rename_axis("month")

,alder_pollen,birch_pollen,grass_pollen,mugwort_pollen
month,,,,
1,0.0,0.0,0.0,0.0
2,0.1,0.0,0.0,0.0
3,5.3,1.2,0.0,0.0
4,0.3,82.9,0.0,0.0
5,0.2,85.7,0.8,0.0
6,0.0,3.7,9.6,0.0
7,0.0,0.1,8.6,2.6
8,0.0,0.0,1.6,1.1
9,0.0,0.0,0.1,0.0


In [ ]:
# 3. One season per pollen type: Stockholm 2025 (small multiples, own y-scale per panel)
import altair as alt

plot = (daily.query("city == 'Stockholm' and day.dt.year == 2025")
        .melt(id_vars="day", value_vars=POLLEN, var_name="pollen", value_name="grains"))
plot["pollen"] = plot["pollen"].str.replace("_pollen", "").str.capitalize()

# Axis labels: values >= 1000 shown as K (e.g. 1.2K)
k_labels = "datum.value >= 1000 ? format(datum.value / 1000, '~r') + 'K' : format(datum.value, '~r')"

(
    alt.Chart(plot)
    .mark_line(strokeWidth=2, color="#2a78d6")
    .encode(
        x=alt.X("day:T", title=None, axis=alt.Axis(format="%b", grid=False)),
        y=alt.Y("grains:Q", title="grains/m³", axis=alt.Axis(labelExpr=k_labels, gridColor="#e1e0d9")),
        tooltip=[alt.Tooltip("day:T", format="%d %b"), "pollen:N", alt.Tooltip("grains:Q", format=".1f")],
    )
    .properties(width=300, height=150)
    .facet(
        facet=alt.Facet("pollen:N", title=None, sort=["Alder", "Birch", "Grass", "Mugwort"],
                        header=alt.Header(labelFontWeight="bold", labelFontSize=12)),
        columns=2,
    )
    .resolve_scale(y="independent")
    .properties(title=alt.Title("Daily pollen, Stockholm 2025", fontWeight="bold",
                                subtitle="Daily mean, local time. Each panel has its own scale"))
    .configure_view(stroke=None)
)

alt.FacetChart(...)

In [ ]:
# 4. Typical daily levels on pollen days (daily mean >= 1 grain/m³), all cities and years
(daily.melt(id_vars=["city", "day"], value_vars=POLLEN, var_name="pollen", value_name="grains")
      .query("grains >= 1")
      .groupby("pollen")["grains"]
      .describe(percentiles=[0.5, 0.75, 0.9, 0.99])
      .round(1))

,count,mean,std,min,50%,75%,90%,99%,max
pollen,,,,,,,,,
alder_pollen,493.0,11.3,33.2,1.0,3.3,7.2,21.9,146.3,508.1
birch_pollen,1864.0,105.3,165.6,1.0,33.0,124.4,331.7,759.9,1404.3
grass_pollen,2683.0,8.4,8.8,1.0,5.1,11.3,20.2,40.9,58.1
mugwort_pollen,1052.0,3.9,3.2,1.0,2.8,5.2,8.2,14.3,27.6


Profiling findings → Phase 2 decisions

- **Ranges OK**: no negative values; weather values are realistic (−32 to 33 °C, rain ≤ 23 mm/h)
- **Off-season = 0**: since 2024 CAMS reports 0 outside the season → fill 2021–2023 NULLs with 0
- **Season months** (2024–2025): alder Mar · birch Apr–May · grass Jun–Jul · mugwort Jul–Aug
- **Levels differ a lot by type** (90th percentile of pollen days): birch ~332 · alder ~22 · grass ~20 · mugwort ~8 grains/m³ → one fixed threshold for all types does not work
- **Proposal for season start/end**: cumulative-sum method (start = day 5% of the year's total pollen is reached, end = 95%), standard in aerobiology and comparable across types
- **"High pollen" days** (for risk KPIs): separate fixed thresholds per type

Decisions for Phase 2 ✅

1. **Empty pollen values → 0.** In 2021–2023 off-season hours are empty (NULL); since 2024 the same model writes 0 there. Empty = "unknown", 0 = "no pollen" → fill with 0 so yearly totals and counts are correct.
2. **Season = 5% → 95% of the year's pollen.** Like a bucket filling up: the season starts when 5% of the year's total has fallen and ends at 95%. Peak = day with the most pollen. Works for every type, big (birch) or small (mugwort).
   - "High pollen" days: separate fixed threshold per type.

CLOSE

In [ ]:
# Close when done, otherwise ingest.py fails with "file is locked"
con.close()

Poniendo la data en dataframe - Python 

Original dataframe

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)

air = con.sql("SELECT * FROM raw.air_quality").df()
weather = con.sql("SELECT * FROM raw.weather").df()
log = con.sql("SELECT * FROM raw.ingestion_log").df()

con.close()  # data is now in the DataFrames; close so dbt / ingest.py can write

In [ ]:
air.info()

<class 'pandas.DataFrame'>
RangeIndex: 302832 entries, 0 to 302831
Data columns (total 11 columns):
 #   Column            Non-Null Count   Dtype         
---  ------            --------------   -----         
 0   city              302832 non-null  str           
 1   time              302832 non-null  datetime64[us]
 2   alder_pollen      229686 non-null  float64       
 3   birch_pollen      226086 non-null  float64       
 4   grass_pollen      239478 non-null  float64       
 5   mugwort_pollen    216438 non-null  float64       
 6   pm2_5             302832 non-null  float64       
 7   pm10              302832 non-null  float64       
 8   ozone             302832 non-null  float64       
 9   nitrogen_dioxide  302832 non-null  float64       
 10  loaded_at         302832 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(8), str(1)
memory usage: 27.4 MB


In [ ]:
air.describe()

,time,alder_pollen,birch_pollen,grass_pollen,mugwort_pollen,pm2_5,pm10,ozone,nitrogen_dioxide,loaded_at
count,302832,229686.000000,226086.000000,239478.000000,216438.000000,302832.000000,302832.000000,302832.000000,302832.000000,302832
mean,2023-11-18 11:30:00,0.611036,20.857903,2.331838,0.512099,4.987334,7.807779,58.436820,6.136922,2026-10-05 11:27:54.401560
min,2021-01-01 00:00:00,0.000000,0.000000,0.000000,0.000000,0.400000,0.500000,0.000000,0.100000,2026-10-05 11:26:22.722817
25%,2022-06-10 17:45:00,0.000000,0.000000,0.000000,0.000000,2.600000,4.300000,47.000000,2.100000,2026-10-05 11:28:06.454277
50%,2023-11-18 11:30:00,0.000000,0.000000,0.000000,0.000000,4.000000,6.600000,59.000000,4.200000,2026-10-05 11:28:12.033715
75%,2025-04-27 05:15:00,0.000000,0.000000,1.000000,0.000000,6.200000,10.000000,70.000000,8.000000,2026-10-05 11:28:15.337506
max,2026-10-04 23:00:00,1760.600000,3060.200000,107.300000,48.000000,95.100000,113.600000,146.000000,72.200000,2026-10-05 11:28:19.388422
std,NaN,12.361282,101.140743,6.601408,1.847808,3.699674,5.081694,17.218528,6.012645,NaN


In [ ]:
# air.city.unique()

<ArrowStringArray>
['Stockholm', 'Malmö', 'Göteborg', 'Uppsala', 'Umeå', 'Luleå']
Length: 6, dtype: str

In [ ]:
weather.info()

<class 'pandas.DataFrame'>
RangeIndex: 302832 entries, 0 to 302831
Data columns (total 8 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   city                  302832 non-null  str           
 1   time                  302832 non-null  datetime64[us]
 2   temperature_2m        302832 non-null  float64       
 3   relative_humidity_2m  302832 non-null  float64       
 4   precipitation         302832 non-null  float64       
 5   wind_speed_10m        302832 non-null  float64       
 6   wind_direction_10m    302832 non-null  float64       
 7   loaded_at             302832 non-null  datetime64[us]
dtypes: datetime64[us](2), float64(5), str(1)
memory usage: 20.5 MB


In [ ]:
weather.describe()

,time,temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m,wind_direction_10m,loaded_at
count,302832,302832.000000,302832.000000,302832.000000,302832.000000,302832.000000,302832
mean,2023-11-18 11:30:00,7.080140,78.946310,0.087605,13.689435,200.924631,2026-10-05 11:27:55.956405
min,2021-01-01 00:00:00,-32.000000,12.000000,0.000000,0.000000,0.000000,2026-10-05 11:26:24.326409
25%,2022-06-10 17:45:00,0.900000,70.000000,0.000000,8.700000,133.000000,2026-10-05 11:28:07.930900
50%,2023-11-18 11:30:00,7.300000,83.000000,0.000000,12.600000,212.000000,2026-10-05 11:28:12.712279
75%,2025-04-27 05:15:00,14.200000,91.000000,0.000000,17.600000,277.000000,2026-10-05 11:28:16.849920
max,2026-10-04 23:00:00,33.100000,100.000000,22.900000,61.000000,360.000000,2026-10-05 11:28:21.685700
std,NaN,8.894449,15.567793,0.363417,6.966990,97.003685,NaN


In [ ]:
weather.city.unique()

<ArrowStringArray>
['Stockholm', 'Malmö', 'Göteborg', 'Uppsala', 'Umeå', 'Luleå']
Length: 6, dtype: str

Step 4 - dbt results (staging + marts)

Rebuild: `cd dbt` → `uv run dbt run` (close the notebook connection first)

In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)
air_clean = con.sql("SELECT * FROM staging.stg_air_quality").df()
con.close()

In [ ]:
air_clean.info()   # all pollen columns: 302832 non-null

<class 'pandas.DataFrame'>
RangeIndex: 302832 entries, 0 to 302831
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   city            302832 non-null  str           
 1   time_utc        302832 non-null  datetime64[us]
 2   time_local      302832 non-null  datetime64[us]
 3   alder_pollen    302832 non-null  float64       
 4   birch_pollen    302832 non-null  float64       
 5   grass_pollen    302832 non-null  float64       
 6   mugwort_pollen  302832 non-null  float64       
 7   pm2_5           302832 non-null  float64       
 8   pm10            302832 non-null  float64       
 9   ozone           302832 non-null  float64       
 10  no2             302832 non-null  float64       
dtypes: datetime64[us](2), float64(8), str(1)
memory usage: 27.4 MB


In [ ]:
# Daily mart: one row per city and day
con = duckdb.connect(DB_PATH, read_only=True)
daily = con.sql("SELECT * FROM marts.daily ORDER BY city, date_local").df()
con.close()

In [ ]:
daily

,city,date_local,alder_pollen_avg,alder_pollen_max,birch_pollen_avg,birch_pollen_max,grass_pollen_avg,grass_pollen_max,mugwort_pollen_avg,mugwort_pollen_max,...,ozone_avg,ozone_max,no2_avg,temperature_avg_c,temperature_min_c,temperature_max_c,humidity_avg_pct,precipitation_mm,wind_speed_avg_kmh,n_hours
0,Umeå,2026-02-15,0.0,0.0,0.00,0.0,0.000000,0.0,0.000000,0.0,...,63.083333,69.0,1.879167,-16.441667,-23.2,-8.1,82.333333,0.0,6.520833,24
1,Umeå,2026-05-09,0.0,0.0,236.95,495.7,0.000000,0.0,0.000000,0.0,...,80.125000,88.0,0.816667,7.608333,4.0,10.6,56.041667,0.0,15.850000,24
2,Umeå,2026-07-07,0.0,0.0,0.00,0.0,8.037500,12.4,0.229167,0.4,...,46.916667,59.0,0.991667,12.691667,8.8,15.6,81.625000,6.3,6.516667,24
3,Umeå,2026-07-08,0.0,0.0,0.00,0.0,4.883333,16.9,0.129167,0.8,...,37.708333,51.0,1.483333,14.116667,9.6,18.5,84.083333,1.5,6.625000,24
4,Umeå,2026-08-22,0.0,0.0,0.00,0.0,0.112500,0.2,0.308333,1.0,...,52.041667,71.0,1.395833,13.375000,6.9,18.8,87.250000,0.4,5.791667,24
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12613,Uppsala,2023-01-24,0.0,0.0,0.00,0.0,0.000000,0.0,0.000000,0.0,...,49.291667,56.0,5.079167,1.358333,-1.6,3.2,91.166667,0.2,15.550000,24
12614,Uppsala,2023-12-29,0.0,0.0,0.00,0.0,0.000000,0.0,0.000000,0.0,...,35.666667,50.0,12.620833,-0.091667,-0.6,0.5,97.333333,5.8,5.425000,24
12615,Uppsala,2023-09-22,0.0,0.0,0.00,0.0,0.000000,0.0,0.166667,0.4,...,67.416667,77.0,4.845833,17.295833,14.4,21.0,86.250000,0.5,16.816667,24
12616,Uppsala,2025-01-01,0.0,0.0,0.00,0.0,0.000000,0.0,0.000000,0.0,...,71.791667,76.0,2.625000,-2.608333,-7.5,-0.8,91.708333,8.4,15.100000,24


In [ ]:
con = duckdb.connect(DB_PATH, read_only=True)
seasons = con.sql("SELECT * FROM marts.seasons ORDER BY pollen_type, year, season_start").df()
con.close()  # close right away so dbt / ingest.py can write

seasons.head()

Step 5 - Data quality (dbt tests)

Run: `cd dbt` → `uv run dbt build` (models + tests) and `uv run dbt source freshness`

| Check | Tables | Result |
|---|---|---|
| No duplicates in the key | raw, staging, all marts | ✅ |
| No empty values (pollen after staging, keys, weather) | staging, hourly, seasons | ✅ |
| No negative values (pollen, pollution, rain, wind) | staging | ✅ |
| Valid ranges (humidity 0–100, wind direction 0–360°, hour 0–23…) | staging, marts | ✅ |
| No missing hours / days | hourly, daily | ✅ |
| Valid cities and pollen types | staging, seasons | ✅ |
| Seasons end after they start | seasons | ✅ |
| Freshness: newest data < 2 days old | raw | ✅ |

**58 tests pass.** Missing-values report: `marts.missing_values` (only expected pollen gaps 2021–2023).

In [ ]:
# Missing-values report: only rows with gaps
con = duckdb.connect(DB_PATH, read_only=True)
missing = con.sql("""
    SELECT source, variable, year, round(avg(pct_missing), 1) AS pct_missing
    FROM marts.missing_values
    WHERE pct_missing > 0
    GROUP BY ALL
    ORDER BY variable, year
""").df()
con.close()

missing

NameError: name 'DB_PATH' is not defined